In [1]:
# 환경 설정

# ================================
# 1. 라이브러리 import
# ================================

import os, math, copy            # 기본 파이썬 유틸리티 라이브러리 (파일 경로, 수학 연산, 객체 복사 등)
import numpy as np               # 수치 계산을 위한 NumPy
import torch                     # PyTorch 딥러닝 프레임워크
from torch.utils.data import DataLoader   # 데이터셋을 배치 단위로 불러오기 위한 DataLoader
import matplotlib.pyplot as plt  # 그래프 시각화를 위한 matplotlib

# HuggingFace 라이브러리
from datasets import load_dataset                                # HuggingFace datasets 로 데이터셋 로드
from transformers import AutoTokenizer, AutoModelForCausalLM     # tokenizer와 language model 자동 로드
from transformers import DataCollatorForLanguageModeling         # LM 학습용 데이터 배치 처리

# ================================
# 2. GPU / CPU 장치 설정
# ================================

# CUDA가 사용 가능하면 GPU 사용, 아니면 CPU 사용
# cuda:3 → 4번째 GPU를 사용한다는 의미 (0부터 시작)
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

# 현재 사용 중인 장치 출력
print("device:", device)

# ================================
# 3. 모델 저장 / 로드 경로 설정
# ================================

# fine-tuning된 모델 저장 경로
save_path = "./distilgpt2-digital"

model_name = "distilbert/distilgpt2"

# fine-tuned 모델이 있으면 그걸 사용
# 없으면 pretrained 모델 사용
load_path_or_name = save_path if os.path.exists(save_path) else model_name

print("loading from:", load_path_or_name)

# ================================
# 4. 입력 시퀀스 길이 설정
# ================================

# 모델에 입력할 최대 토큰 길이
# 긴 문장은 이 길이로 잘려서 사용됨
max_length = 256

# 평가 시 사용할 텍스트 샘플 수 제한
# 전체 dataset 대신 일부만 사용하여 평가 속도 향상
eval_text_limit = 2000

# 평가 시 batch 크기
# GPU 메모리 사용량과 속도에 영향
eval_batch_size = 8

# ================================
# 5. 노이즈 실험 설정
# ================================

# 모델 weight 또는 activation에 주입할 Gaussian noise 수준
# 예: 0.01 → 표준편차가 0.01인 노이즈
# AIMC / analog hardware simulation 실험에서 자주 사용
noise_levels = [0, 0.01, 0.02, 0.03, 0.04, 0.05]

# 0      : 노이즈 없음 (baseline)
# 0.01~0.05 : 점점 강한 노이즈 환경에서 모델 성능 측정
# 보통 PPL(perplexity) 변화로 robustness 평가   

/home/hyun/miniconda3/envs/al/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda:3
loading from: ./distilgpt2-digital


In [2]:
# ================================
# 1. 평가에 사용할 데이터셋 불러오기
# ================================

# HuggingFace datasets 라이브러리를 이용해 WikiText-2 데이터셋 로드
# 이 데이터셋은 언어 모델의 성능 평가(PPL 등)에 자주 사용됨
raw = load_dataset("wikitext", "wikitext-2-raw-v1")


# ================================
# 2. 토크나이저(tokenizer) 불러오기
# ================================

# 모델과 동일한 tokenizer를 로드
# tokenizer는 문장을 모델이 이해할 수 있는 숫자 토큰으로 변환하는 역할을 함
tokenizer = AutoTokenizer.from_pretrained(load_path_or_name, use_fast=True)

# 일부 모델은 pad_token이 정의되어 있지 않기 때문에
# 그런 경우 문장 끝 토큰(eos_token)을 padding 토큰으로 사용
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ================================
# 3. 텍스트를 토큰으로 변환하는 함수 정의
# ================================

def tokenize_fn(examples):
    # 문장을 토큰으로 변환
    # truncation=True → 최대 길이를 넘으면 잘라냄
    # max_length → 문장의 최대 토큰 길이 제한
    return tokenizer(examples["text"], truncation=True, max_length=max_length)


# ================================
# 4. 평가 데이터 전처리
# ================================

# validation 데이터 중 일부(eval_text_limit 만큼)만 사용
# → 평가 속도를 빠르게 하기 위한 설정
eval_ds = raw["validation"].select(
    range(min(len(raw["validation"]), eval_text_limit))
).map(
    tokenize_fn,                 # 위에서 만든 토큰화 함수 적용
    batched=True,                # 여러 문장을 한번에 처리
    remove_columns=raw["validation"].column_names  # 기존 텍스트 컬럼 제거
)

# 데이터 타입을 PyTorch tensor 형식으로 변환
# → 모델에 바로 입력할 수 있도록 설정
eval_ds.set_format("torch")


# ================================
# 5. 언어모델용 데이터 배치 처리 설정
# ================================

# DataCollatorForLanguageModeling
# → 배치를 만들 때 padding을 자동으로 처리해주는 도구
# mlm=False → GPT 계열 모델은 Masked LM이 아닌
#             Causal Language Modeling 방식 사용
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


# ================================
# 6. DataLoader 생성
# ================================

# DataLoader는 데이터를 batch 단위로 모델에 공급하는 역할
eval_loader = DataLoader(
    eval_ds,
    batch_size=eval_batch_size,  # 한 번에 처리할 데이터 개수
    shuffle=False,               # 평가이므로 데이터 순서 섞지 않음
    collate_fn=collator          # 배치 생성 방식
)


# ================================
# 7. 평가 데이터 크기 확인
# ================================

# 실제 평가에 사용되는 문장 개수 출력
print("eval size:", len(eval_ds))

eval size: 2000


In [3]:
# ================================
# Perplexity(PPL) 평가 함수 정의
# ================================

# 평가 과정에서는 gradient 계산이 필요 없으므로 비활성화
# 메모리 사용 감소 + 계산 속도 향상
torch.set_grad_enabled(False)

def eval_ppl(model, dataloader, device):

    # 모델을 평가 모드로 설정
    # (dropout 등 학습 전용 기능 비활성화)
    model.eval()

    # 전체 loss와 전체 토큰 수를 저장할 변수
    total_loss = 0.0
    total_tokens = 0

    # DataLoader에서 batch 단위로 데이터 가져오기
    for batch in dataloader:

        # batch 안의 모든 데이터를 GPU 또는 CPU로 이동
        batch = {k: v.to(device) for k, v in batch.items()}

        # 모델에 입력 데이터를 넣어 출력 계산
        out = model(
            input_ids=batch["input_ids"],               # 모델 입력 토큰
            attention_mask=batch.get("attention_mask", None),  # 실제 토큰 위치 표시
            labels=batch["labels"],                     # 정답 토큰 (loss 계산에 사용)
        )

        # 모델이 계산한 loss 값 가져오기
        loss = out.loss

        # 실제 토큰 개수 계산
        # attention_mask가 있으면 padding을 제외한 실제 토큰 수 사용
        if "attention_mask" in batch:
            n_tok = int(batch["attention_mask"].sum().item())
        else:
            # attention_mask가 없으면 전체 토큰 수 사용
            n_tok = batch["input_ids"].numel()

        # 전체 loss 누적
        # (batch loss × 토큰 수)
        total_loss += float(loss.item()) * n_tok

        # 전체 토큰 수 누적
        total_tokens += n_tok

    # 평균 negative log likelihood 계산
    avg_nll = total_loss / max(total_tokens, 1)

    # perplexity 계산
    # ppl = exp(avg_nll)
    ppl = math.exp(avg_nll)

    # 평균 loss와 perplexity 반환
    return avg_nll, ppl

print("Perplexity evaluation function is successfully configured.")

Perplexity evaluation function is successfully configured.


In [4]:
# 레이어 그룹 정의
layer_groups = {
    # per-layer targets (transformer.h.{i}. 아래)
    "ln_1":        "per_layer",
    "attn.c_attn": "per_layer",
    "attn.c_proj": "per_layer",
    "ln_2":        "per_layer",
    "mlp.c_fc":    "per_layer",
    "mlp.c_proj":  "per_layer",

    # global targets (layer_idx 없음)
    "transformer.wte": "global",
    "transformer.wpe": "global",
    "lm_head":         "global",
}

In [5]:
# 노이즈 주입 함수 (1) - 레이어 이름 별로 분리하기
def inject_noise_to_named_modules(model, name_prefixes, noise_std):
    """
    model.named_modules()의 name이 prefix 중 하나로 시작하면 그 모듈의 weight에 노이즈 주입
    - noise_std=0이면 아무 변화 없음
    - Per-channel max scaled Gaussian (row-wise scale)
    """
    if noise_std == 0:
        return

    with torch.no_grad():
        for name, module in model.named_modules(): # 모델 안의 모든 서브모듈을 (이름, 모듈) 형태로 순회함
            if not hasattr(module, "weight"):
                continue

            if not any(name.startswith(pfx) for pfx in name_prefixes):
                continue

            w = module.weight.data
            if w.ndim >= 2:
                # row-wise scale: max over in_features
                max_w = torch.max(torch.abs(w), dim=1, keepdim=True).values
                noise = torch.randn_like(w) * noise_std * max_w # 가우시안 정규 분포 노이즈
                w.add_(noise)
            elif w.ndim == 1: 
                max_w = torch.max(torch.abs(w))
                w.add_(torch.randn_like(w) * noise_std * max_w)

In [6]:
# 노이즈 주입 함수(2) - 특정 븝록 i의 특정 그룹(attn/mlp/all) 에만 노이즈 주입하기
def get_prefixes_for_layer_group(layer_idx, group_key):
    """
    group_key에 해당하는 named_modules() prefix 리스트를 반환.
    - per-layer: transformer.h.{layer_idx}.xxx
    - global: layer_idx 무시, transformer.wte / transformer.wpe / lm_head
    """
    if group_key in ["transformer.wte", "transformer.wpe", "lm_head"]:
        return [group_key]  # 전역 모듈

    base = f"transformer.h.{layer_idx}."
    if group_key == "ln_1":
        return [base + "ln_1"]
    elif group_key == "attn.c_attn":
        return [base + "attn.c_attn"]
    elif group_key == "attn.c_proj":
        return [base + "attn.c_proj"]
    elif group_key == "ln_2":
        return [base + "ln_2"]
    elif group_key == "mlp.c_fc":
        return [base + "mlp.c_fc"]
    elif group_key == "mlp.c_proj":
        return [base + "mlp.c_proj"]
    else:
        raise ValueError(f"Unknown group_key: {group_key}")


In [7]:
# digital baseline 측정
base_model = AutoModelForCausalLM.from_pretrained(load_path_or_name).to(device)
base_nll, base_ppl = eval_ppl(base_model, eval_loader, device)

n_layers = len(base_model.transformer.h)
print(f"Digital baseline PPL: {base_ppl:.2f} | n_layers={n_layers}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Digital baseline PPL: 35.35 | n_layers=6


In [8]:
#레이어 별 측정 및 테스트
import collections
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM

# -------------------------
# 반복 횟수 설정 (많을수록 정교하지만 실험 시간이 오래 걸린다.)
# -------------------------
n_trials = 5
rows = []
results = {}  # (op_type, layer) -> dict(noise -> (mean_ppl, std_ppl)) 레이어 종류(op_type)와 레이어 번호(layer)를 기준으로 결과를 저장하고, 그 안에 노이즈 크기별 PPL 결과를 기록

# (선택) 재현성: trial마다 seed를 고정하고 싶으면 사용
base_seed = 1234

# -------------------------
# baseline 모델의 파라미터 state_dict를 미리 저장
# -------------------------
base_model = AutoModelForCausalLM.from_pretrained(load_path_or_name).to(device)
base_model.eval()
base_sd = {k: v.detach().cpu() for k, v in base_model.state_dict().items()}  # CPU에 보관

# baseline PPL(노이즈 0 기준선)도 미리 구해두면 ΔPPL 계산 쉬움
with torch.inference_mode():
    _, base_ppl = eval_ppl(base_model, eval_loader, device)

print("Digital baseline PPL:", base_ppl)


# -------------------------
# 스윕 실행 (per-layer / global 분리)
# -------------------------
per_layer_keys = [k for k, v in layer_groups.items() if v == "per_layer"]
global_keys    = [k for k, v in layer_groups.items() if v == "global"]

for group_key in per_layer_keys:
    for layer_idx in range(n_layers):
        prefixes = get_prefixes_for_layer_group(layer_idx, group_key)

        for noise_std in noise_levels:
            trial_ppls = []

            for t in range(n_trials):
                torch.manual_seed(base_seed + t)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(base_seed + t)

                # 깨끗한 모델 생성 + 로드
                model_i = AutoModelForCausalLM.from_pretrained(load_path_or_name).to(device)
                model_i.load_state_dict(base_sd, strict=True)
                model_i.eval()

                # 노이즈 주입
                inject_noise_to_named_modules(model_i, prefixes, noise_std)

                # PPL 측정
                with torch.inference_mode():
                    _, ppl = eval_ppl(model_i, eval_loader, device)

                trial_ppls.append(float(ppl))

            mean_ppl = float(np.mean(trial_ppls))
            std_ppl  = float(np.std(trial_ppls))

            print(f"[{group_key} | Layer {layer_idx:02d} | noise {noise_std:.2f}] "
                  f"PPL: {mean_ppl:.2f} (±{std_ppl:.2f})")

            rows.append({
                "layer": int(layer_idx),
                "noise": float(noise_std),
                "op_type": group_key,
                "ppl_mean": mean_ppl,
                "ppl_std": std_ppl,
                "delta_ppl_mean": mean_ppl - base_ppl,
            })

            results[(group_key, layer_idx, noise_std)] = (mean_ppl, std_ppl)


# global targets: layer_idx = -1 같은 더미값으로 저장
GLOBAL_LAYER_ID = -1

for group_key in global_keys:
    prefixes = get_prefixes_for_layer_group(None, group_key)

    for noise_std in noise_levels:
        trial_ppls = []

        for t in range(n_trials):
            torch.manual_seed(base_seed + t)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(base_seed + t)

            model_i = AutoModelForCausalLM.from_pretrained(load_path_or_name).to(device)
            model_i.load_state_dict(base_sd, strict=True)
            model_i.eval()

            inject_noise_to_named_modules(model_i, prefixes, noise_std)

            with torch.inference_mode():
                _, ppl = eval_ppl(model_i, eval_loader, device)

            trial_ppls.append(float(ppl))

        mean_ppl = float(np.mean(trial_ppls))
        std_ppl  = float(np.std(trial_ppls))

        print(f"[{group_key} | GLOBAL | noise {noise_std:.2f}] "
              f"PPL: {mean_ppl:.2f} (±{std_ppl:.2f})")

        rows.append({
            "layer": int(GLOBAL_LAYER_ID),
            "noise": float(noise_std),
            "op_type": group_key,
            "ppl_mean": mean_ppl,
            "ppl_std": std_ppl,
            "delta_ppl_mean": mean_ppl - base_ppl,
        })

        results[(group_key, GLOBAL_LAYER_ID, noise_std)] = (mean_ppl, std_ppl)


df = pd.DataFrame(rows)
df.head()

Digital baseline PPL: 35.353949596653855
[ln_1 | Layer 00 | noise 0.00] PPL: 35.35 (±0.00)
[ln_1 | Layer 00 | noise 0.01] PPL: 35.35 (±0.00)
[ln_1 | Layer 00 | noise 0.02] PPL: 35.36 (±0.00)
[ln_1 | Layer 00 | noise 0.03] PPL: 35.36 (±0.01)
[ln_1 | Layer 00 | noise 0.04] PPL: 35.37 (±0.01)
[ln_1 | Layer 00 | noise 0.05] PPL: 35.39 (±0.01)
[ln_1 | Layer 01 | noise 0.00] PPL: 35.35 (±0.00)
[ln_1 | Layer 01 | noise 0.01] PPL: 35.37 (±0.02)
[ln_1 | Layer 01 | noise 0.02] PPL: 35.41 (±0.04)
[ln_1 | Layer 01 | noise 0.03] PPL: 35.46 (±0.06)
[ln_1 | Layer 01 | noise 0.04] PPL: 35.54 (±0.10)
[ln_1 | Layer 01 | noise 0.05] PPL: 35.64 (±0.15)
[ln_1 | Layer 02 | noise 0.00] PPL: 35.35 (±0.00)
[ln_1 | Layer 02 | noise 0.01] PPL: 35.38 (±0.02)
[ln_1 | Layer 02 | noise 0.02] PPL: 35.43 (±0.06)
[ln_1 | Layer 02 | noise 0.03] PPL: 35.50 (±0.12)
[ln_1 | Layer 02 | noise 0.04] PPL: 35.60 (±0.19)
[ln_1 | Layer 02 | noise 0.05] PPL: 35.73 (±0.27)
[ln_1 | Layer 03 | noise 0.00] PPL: 35.35 (±0.00)
[ln_1 | L

,layer,noise,op_type,ppl_mean,ppl_std,delta_ppl_mean
0,0,0.00,ln_1,35.353950,0.000000,0.000000
1,0,0.01,ln_1,35.353984,0.002246,0.000035
2,0,0.02,ln_1,35.357318,0.004040,0.003368
3,0,0.03,ln_1,35.364028,0.005318,0.010078
4,0,0.04,ln_1,35.374262,0.005991,0.020313


In [10]:
# 결과를 데이터프레임 형태로 변환
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GLOBAL_LAYER_ID = -1

# ---- 1) results dict -> df_plot (float noise 키 이슈 방지 포함)
def results_dict_to_df_plot(results, noise_levels):
    rows = []
    keys_ol = sorted(set((op_type, layer) for (op_type, layer, noise) in results.keys()))

    for (op_type, layer) in keys_ol:
        for nz in noise_levels:
            val = results.get((op_type, layer, float(nz)), None)

            if val is None:
                # float 키 오차 대비: 가장 가까운 noise로 매칭
                candidates = [(k, v) for (k, v) in results.items() if k[0] == op_type and k[1] == layer]
                noises = [k[2] for (k, _) in candidates]
                nearest = min(noises, key=lambda x: abs(x - float(nz)))
                mean_ppl, std_ppl = results[(op_type, layer, nearest)]
            else:
                mean_ppl, std_ppl = val

            rows.append({
                "layer": int(layer),
                "noise": float(nz),
                "op_type": str(op_type),
                "ppl": float(mean_ppl),
                "ppl_std": float(std_ppl),
            })

    return pd.DataFrame(rows)

df_plot = results_dict_to_df_plot(results, noise_levels)
df_plot.head()


,layer,noise,op_type,ppl,ppl_std
0,0,0.00,attn.c_attn,35.353950,0.000000
1,0,0.01,attn.c_attn,35.492034,0.044856
2,0,0.02,attn.c_attn,36.038029,0.332522
3,0,0.03,attn.c_attn,37.018301,0.861439
4,0,0.04,attn.c_attn,38.265849,1.210882


In [12]:
# 결과 파일을 따로 저장
import os
import json
import pickle

# ================================
# 저장 폴더 만들기
# ================================
save_dir = "./noise_experiment_outputs"
os.makedirs(save_dir, exist_ok=True)

# 파일 경로
df_path = os.path.join(save_dir, "df_results.csv")
df_plot_path = os.path.join(save_dir, "df_plot.csv")
meta_path = os.path.join(save_dir, "metadata.json")
results_path = os.path.join(save_dir, "results_dict.pkl")

# ================================
# 1. 테이블 형태 저장
# ================================
df.to_csv(df_path, index=False, encoding="utf-8-sig")
df_plot.to_csv(df_plot_path, index=False, encoding="utf-8-sig")

# ================================
# 2. 메타데이터 저장
# ================================
metadata = {
    "model_path": load_path_or_name,
    "base_ppl": float(base_ppl),
    "n_layers": int(n_layers),
    "noise_levels": [float(x) for x in noise_levels],
    "eval_text_limit": int(eval_text_limit),
    "eval_batch_size": int(eval_batch_size),
    "max_length": int(max_length),
    "n_trials": int(n_trials),
    "base_seed": int(base_seed),
    "layer_groups": layer_groups,
    "global_layer_id": int(GLOBAL_LAYER_ID),
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

# ================================
# 3. results 딕셔너리 원본 저장
# ================================
with open(results_path, "wb") as f:
    pickle.dump(results, f)

print("Saved files:")
print("df       ->", df_path)
print("df_plot  ->", df_plot_path)
print("metadata ->", meta_path)
print("results  ->", results_path)

Saved files:
df       -> ./noise_experiment_outputs/df_results.csv
df_plot  -> ./noise_experiment_outputs/df_plot.csv
metadata -> ./noise_experiment_outputs/metadata.json
results  -> ./noise_experiment_outputs/results_dict.pkl
